In [ ]:
# Import Liabrary
from typing import Optional, List, Type
from pydantic import BaseModel, ValidationError, Field, validator
import pandas as pd
import geopandas as gpd
from shapely import wkb, Point
from shapely.wkt import loads as wkt_loads

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from huggingface_hub import HfApi, hf_hub_download
import inspect
import os
import io

In [ ]:
def crs_detector(gdf):
    crs = "EPSG:4326"
    for geom in gdf["geometry"]:
        if geom is not None and not geom.is_empty:
            minx, miny, maxx, maxy = geom.bounds
            if -180 <= minx <= 180 and -180 <= maxx <= 180 and -90 <= miny <= 90 and -90 <= maxy <= 90:
                crs="EPSG:4326"
            else:
                crs="EPSG:7855" # VicGrid GDA2020
    return crs

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
import geopandas as gpd
import os

# Hugging Face dataset repository identifier
HF_REPO_ID = "Loveffort/Capstone-dataset"
# Load HF access token from environment variable, fallback to static token
HF_TOKEN = os.getenv("HF_TOKEN")
# Flag to judge whether runtime environment is Google Colab
IS_COLAB = "COLAB_GPU" in os.environ
# Define working directory for cache & local data storage
WORK_DIR = "/content/victoria_fire_risk_data" if IS_COLAB else "./victoria_fire_risk_data"
# Create target directory if it does not exist
os.makedirs(WORK_DIR, exist_ok=True)v

class HFDataManager:
    def __init__(self, repo_id, token, work_dir):
        """
        Initialize Hugging Face dataset file manager
        :param repo_id: Target HF dataset repository path
        :param token: Hugging Face authentication access token
        :param work_dir: Local directory for file cache and storage
        """
        self.repo_id = repo_id
        self.token = token
        self.work_dir = work_dir
        # Initialize HF API client only if valid token is provided
        self.api = HfApi(token=token) if token else None

    def load(self, hf_file_path):
        """
        Download and parse dataset file from Hugging Face Hub
        Support formats: Shapefile, Parquet, GeoJSON, CSV, Excel(.xlsx/.xls)
        :param hf_file_path: Relative file path inside HF dataset repo
        :return: Parsed GeoDataFrame / DataFrame object
        """
        # Handle multi-file shapefile package
        if hf_file_path.lower().endswith(".shp"):
            base = hf_file_path.replace(".shp", "")
            local_files = {}
            # Required auxiliary extensions for complete shapefile
            ext_list = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
            for ext in ext_list:
                filename = f"{base}{ext}"
                try:
                    local_file = hf_hub_download(
                        repo_id=self.repo_id,
                        repo_type="dataset",
                        filename=filename,
                        token=self.token,
                        cache_dir=self.work_dir
                    )
                    local_files[ext] = local_file
                except Exception as e:
                    print(f"Warning: {filename} not found ({e})")
            # Raise error if core .shp file missing
            shp_path = local_files.get(".shp")
            if not shp_path:
                raise FileNotFoundError("Shapefile .shp not found in repo")
            return gpd.read_file(shp_path)

        # Download single standalone file from HF hub
        local_file = hf_hub_download(
            repo_id=self.repo_id,
            repo_type="dataset",
            filename=hf_file_path,
            token=self.token,
            cache_dir=self.work_dir
        )

        # Parse file by extension type
        if hf_file_path.lower().endswith(".parquet"):
            return pd.read_parquet(local_file)
        elif hf_file_path.lower().endswith(".geojson"):
            return gpd.read_file(local_file)
        elif hf_file_path.lower().endswith(".csv"):
            return gpd.read_file(local_file, driver="CSV")
        elif hf_file_path.lower().endswith(".xlsx") or hf_file_path.lower().endswith(".xls"):
            return pd.read_excel(local_file)
        else:
            raise ValueError(f"Unsupported file format: {hf_file_path}")

    def save_gdf(self, gdf, file_name, format="parquet"):
        """
        Save GeoDataFrame locally then upload file to Hugging Face dataset repo
        Supported output formats: parquet, csv, geojson
        :param gdf: Target GeoDataFrame to export & upload
        :param file_name: Output filename without extension
        :param format: Target file storage format
        """
        if not self.api:
            raise ValueError("HF Token not configured!")

        file_fullname = f"{file_name}.{format}"
        local_path = os.path.join(self.work_dir, file_fullname)

        # Export GeoDataFrame to local file with specified format
        if format == "parquet":
            gdf.to_parquet(local_path, engine="pyarrow", index=False)
        elif format == "csv":
            gdf.to_csv(local_path, index=False)
        elif format == "geojson":
            gdf.to_file(local_path, driver="GeoJSON")
        else:
            raise ValueError("Unsupported file format")

        # Upload generated local file to HF dataset repository
        self.api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=file_fullname,
            repo_id=self.repo_id,
            repo_type="dataset",
            commit_message=f"Save {file_fullname}"
        )
        print(f"Uploaded to HF Hub: {self.repo_id}/{file_fullname}")

In [ ]:
## Modified filters to support fuzzy matching and conditional matching.
## Default to projected CRS (VicGrid GDA2020)
class Record(BaseModel):
    X: Optional[float] = Field(None, description="X coordinate")
    Y: Optional[float] = Field(None, description="Y coordinate")
    gnaf_confidence: Optional[float] = Field(None, ge=0, le=1, description="GNAF confidence score (-1，0，1，2)")
    distance_to_gnaf: Optional[float] = Field(None, ge=0, description="Distance to GNAF point in meters")
    gnaf_lat: Optional[float] = Field(None, ge=-90, le=90, description="GNAF latitude")
    gnaf_long: Optional[float] = Field(None, ge=-180, le=180, description="GNAF longitude")
    geom: Optional[str] = Field(None, description="Geometry in WKT format")
    bpa_areaha: Optional[float] = Field(None, ge=0, description="Burned area in hectares")
    allcount: Optional[int] = Field(None, ge=0, description="Total count")
    yrsfrburn: Optional[int] = Field(None, ge=0, description="Years since last burn")
    firecount: Optional[int] = Field(None, ge=0, description="Fire count")
    burncount: Optional[int] = Field(None, ge=0, description="Burn count")
    zonetype: Optional[float] = Field(None, description="Zone type code")


def read_select(
    path: str,
    model_class: Type[BaseModel],
    usecols=None,
    filters=None,
    xy_cols=None,
    wkb_col=None,
    crs="EPSG:7855",  # Target: Projected CRS
    encoding=None
):
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    if encoding:
        encodings_to_try = [encoding] + encodings_to_try

    # Read CSV with required columns
    df = None
    for enc in encodings_to_try:
        try:
            columns_to_read = usecols.copy() if usecols else []
            if filters:
                columns_to_read.extend(filters.keys())
                columns_to_read = list(set(columns_to_read))  # Remove duplicates
            df = pd.read_csv(path, usecols=columns_to_read, encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise ValueError(f"Cannot read file: {path}")


    # Apply flexible filters (supports both functions and raw values)
    if filters:
        for col, filter_val in filters.items():
            if col not in df.columns:
                raise ValueError(f"Filter column '{col}' not found in data")

            # Case 1: If filter is a function (e.g., lambda for fuzzy/numerical match)
            if callable(filter_val):
                df = df[filter_val(df[col])]  # Apply function to column

            # Case 2: If filter is a raw value (exact match)
            else:
                df = df[df[col] == filter_val]  # Exact match


        # Drop filter columns not in usecols
        cols_to_drop = [col for col in filters.keys() if (usecols and col not in usecols)]
        df = df.drop(columns=cols_to_drop, errors="ignore")

    # Remove rows with missing values in target columns
    df = df.dropna(subset=usecols) if usecols else df

    # Convert to Pydantic records
    records: List[BaseModel] = []
    for row in df.to_dict(orient="records"):
        filtered_row = {k: v for k, v in row.items() if k in model_class.model_fields}
        try:
            records.append(model_class(**filtered_row))
        except Exception as e:
            pass  # Skip invalid rows

    # Convert to GeoDataFrame if needed
    if xy_cols:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xy_cols[0]], df[xy_cols[1]]), crs="EPSG:4326")
        gdf = gdf.to_crs(crs)
    elif wkb_col:
        df["geometry"] = df[wkb_col].apply(lambda x: wkb.loads(bytes.fromhex(x), hex=True))
        temp_gdf = gpd.GeoDataFrame(df, geometry="geometry")
        original_crs = crs_detector(temp_gdf)
        gdf = gpd.GeoDataFrame(df.drop(columns=[wkb_col]), geometry="geometry",crs=original_crs)
        gdf=gdf.to_crs(crs)
    else:
        gdf = df

    columns = [c for c in df.columns if c != "geometry"]
    return records, gdf, columns

In [ ]:
metro_records, metro_gdf, metro_cols= read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Metro_Fire_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

rural_records, rural_gdf, rural_cols= read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Rural_Country_Fire_Service_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

bushfire_records, bushfire_gdf,bushfire_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_bushfire_prone_area.csv",
    model_class=Record,
    usecols=["geom","bpa_areaha"],
    wkb_col="geom"
)

fire_history_records, fire_history_gdf, fire_history_cols  = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_history_latest.csv",
    model_class=Record,
    usecols=["geom","allcount","yrsfrburn","firecount","burncount"],
    wkb_col="geom"
)

fire_manage_records, fire_manage_gdf,fire_manage_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
    model_class=Record,
    usecols=["geom","zonetype"],
    wkb_col="geom"
)

renewable_records, renewable_gdf, renewable_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/renewable_project_nem.csv",
    model_class=Record,
    filters={"Region":"VIC1"},
    usecols=["X","Y"],
    xy_cols=("X","Y"),
    encoding='latin-1'
)

file_path = "/content/drive/My Drive/vic_native_vegetation.csv"
native_veg_records, native_veg_gdf, native_veg_cols = read_select(
    path=file_path,
    model_class=Record,
    usecols=["geom","xgroupname", "areasqm", "evc_bcs", "evc_mut"],
    wkb_col="geom"
)

In [ ]:
# ---- fire facilities ----
hf_manager.save_gdf(metro_gdf)
hf_manager.save_gdf(rural_gdf)

# ---- bushfire & history ----
hf_manager.save_gdf(bushfire_gdf)
hf_manager.save_gdf(fire_history_gdf)
hf_manager.save_gdf(fire_manage_gdf)

# ---- renewable ----
hf_manager.save_gdf(renewable_gdf)

# ---- native_veg ----
hf_manager.save_gdf(native_veg_gdf)

In [ ]:
property_gdf = hf_manager.load_geo_data("Property/PROPERTY_VIEW.shp")
property_gdf = property_gdf[['PFI', 'geometry']].set_index('PFI')
print("Original CRS:", property_gdf.crs)
property_gdf=property_gdf.to_crs("EPSG:7855")
print(property_gdf.shape)
print(property_gdf.head())

In [ ]:
# --------------------------
# Step 1: Filter candidate properties within 5km of substations
# --------------------------
file_path = "DAPR_summary_v2.xlsx"
substation_df = hf_manager.load(file_path)
lat_col = "latitude"
lon_col = "longitude"
required_cols = {lat_col, lon_col}
missing_cols = [col for col in required_cols if col not in substation_df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in DAPR_summary.xlsx: {missing_cols}")

# Keep all original xlsx columns; only latitude/longtitude are used to build geometry
substation_df = substation_df.dropna(subset=[lat_col, lon_col]).copy()
substation_gdf = gpd.GeoDataFrame(
    substation_df,
    geometry=gpd.points_from_xy(substation_df[lon_col], substation_df[lat_col]),
    crs="EPSG:4326",
)
hf_manager.save_gdf(substation_gdf)

In [ ]:
def filter_candidate_properties(substation_gdf, property_gdf, max_distance_km=5):
    """
    Filter property polygons within distance threshold of substations, retain nearest substation per property
    Args:
        substation_gdf: GeoDataFrame of substation point features, must contain "Substation" column
        property_gdf: GeoDataFrame of property polygon blocks with valid CRS
        max_distance_km: Maximum search radius between property centroid and substation point
    Returns:
        GeoDataFrame with columns [PFI, geometry, Substation], original input CRS
    """
    # Add unique PFI index column for each property block
    property_with_pfi = property_gdf.reset_index(names="PFI").copy()

    # Validate required column in substation dataset
    if "Substation" not in substation_gdf.columns:
        raise KeyError("substation_gdf must contain column 'substation' for output naming")

    # Validate coordinate reference system exists
    if property_with_pfi.crs is None:
        raise ValueError("property_gdf CRS is missing; cannot perform distance-based spatial matching")

    # Convert to projected CRS for meter-based distance calculation
    if property_with_pfi.crs.is_geographic:
        projected_crs = property_with_pfi.estimate_utm_crs() or "EPSG:3857"
    else:
        projected_crs = property_with_pfi.crs

    # Reproject both datasets to consistent planar CRS
    property_projected = property_with_pfi.to_crs(projected_crs)
    substation_projected = substation_gdf.to_crs(projected_crs)

    # Calculate polygon centroid as representative point of each property
    property_projected["centroid"] = property_projected.geometry.centroid
    max_distance_m = max_distance_km * 1000
    candidate_list = []

    # Iterate each substation to find surrounding matching properties
    for _, station_row in substation_projected.iterrows():
        station_geom = station_row.geometry
        substation_name = station_row["Substation"]

        # Compute straight-line distance from property centroid to current substation
        property_projected["distance_to_substation_m"] = property_projected["centroid"].distance(station_geom)

        # Filter properties within distance threshold
        nearby_properties = property_projected[
            property_projected["distance_to_substation_m"] <= max_distance_m
        ].copy()
        nearby_properties["Substation"] = substation_name
        # Retain only required output fields
        nearby_properties = nearby_properties[["PFI", "geometry", "Substation", "distance_to_substation_m"]]

        candidate_list.append(nearby_properties)
        print(f"Processed substation {substation_name}, found {len(nearby_properties)} nearby properties")

    # Return empty standardized GeoDataFrame if no matching records
    if not candidate_list:
        return gpd.GeoDataFrame(
            columns=["PFI", "geometry", "Substation"],
            geometry="geometry",
            crs=property_with_pfi.crs,
        )

    # Combine all matched records across all substations
    all_candidates = pd.concat(candidate_list, ignore_index=True)
    # Keep only nearest substation for each unique property PFI
    nearest_idx = all_candidates.groupby("PFI")["distance_to_substation_m"].idxmin()
    nearest_candidates = all_candidates.loc[nearest_idx].copy()

    # Convert back to original geographic CRS and trim output columns
    candidate_gdf = gpd.GeoDataFrame(
        nearest_candidates[["PFI", "geometry", "Substation"]],
        geometry="geometry",
        crs=projected_crs,
    ).to_crs(property_with_pfi.crs)

    return candidate_gdf[["PFI", "geometry", "Substation"]]

In [ ]:
df = station_property_gdf.copy()
df["PFI"] = pd.to_numeric(df["PFI"], errors="coerce").astype("Int64")
df = df.dropna(subset=["PFI"])

station_property_gdf_u = (
    df.sort_values("PFI")
      .drop_duplicates(subset=["PFI"], keep="first")
      .reset_index(drop=True)
)
station_property_gdf = station_property_gdf_u[["PFI", "geometry", "Substation"]]
hf_manager.save_gdf(station_property_gdf)

In [ ]:
# ---- fire facilities ----
metro_gdf = hf_manager.load("metro_gdf.parquet")
rural_gdf = hf_manager.load("rural_gdf.parquet")

# ---- bushfire & history ----
bushfire_gdf = hf_manager.load("bushfire_gdf.parquet")
fire_history_gdf = hf_manager.load("fire_history_gdf.parquet")
fire_manage_gdf = hf_manager.load("fire_manage_gdf.parquet")

# ---- renewable ----
renewable_gdf = hf_manager.load("renewable_gdf.parquet")

# ---- native_veg ----
native_veg_gdf = hf_manager.load("native_veg_gdf.parquet")

# ---- property ----
station_property_gdf = hf_manager.load("station_property_gdf.parquet")

In [ ]:
print(metro_gdf)
print(rural_gdf)
print(bushfire_gdf)
print(fire_history_gdf)
print(fire_manage_gdf)
print(renewable_gdf)
print(native_veg_gdf)
print(station_property_gdf)

In [ ]:
from tqdm import tqdm

tqdm.pandas()

def produce_geo_features(
    station_property_gdf: gpd.GeoDataFrame,
    row_func,
    desc: str,
    index_col: str = "PFI",
):
    base = station_property_gdf[["PFI", "geometry"]].copy()

    tqdm.pandas(desc=desc)
    out = base.progress_apply(lambda row: row_func(row), axis=1).tolist()
    df = pd.DataFrame(out)

    if index_col not in df.columns:
        df[index_col] = base[index_col].values

    return df

In [ ]:
# --------------------------
# 2.1 Construct fire-facility feature sub-vectors
# Quantify the number and closest distance of fire-facility facilities within 5km to reflect the ease of rescue.
# --------------------------
radius = 5000
metro_sindex = metro_gdf.sindex
rural_sindex = rural_gdf.sindex

def fire_facility_row_func(row):
    centroid = row["geometry"].centroid
    total = 0
    min_distance = float("inf")

    # --- metro ---
    metro_bounds = centroid.buffer(radius).bounds
    midx = list(metro_sindex.intersection(metro_bounds))
    if midx:
        metro_cands = metro_gdf.iloc[midx]
        for g in metro_cands.geometry:
            d = g.distance(centroid)
            if d <= radius:
                total += 1
                if d < min_distance:
                    min_distance = d

    # --- rural ---
    rural_bounds = centroid.buffer(radius).bounds
    ridx = list(rural_sindex.intersection(rural_bounds))
    if ridx:
        rural_cands = rural_gdf.iloc[ridx]
        for g in rural_cands.geometry:
            d = g.distance(centroid)
            if d <= radius:
                total += 1
                if d < min_distance:
                    min_distance = d

    if min_distance == float("inf"):
        min_distance = radius * 2

    return {
        "PFI": row["PFI"],
        "total_facilities_5km": total,
        "closest_facility_distance": float(min_distance),
    }

fire_facility_features = produce_geo_features(
    station_property_gdf,
    fire_facility_row_func,
    desc="Processing fire facilities"
)

hf_manager.save_gdf(fire_facility_features)

In [ ]:
# --------------------------
# 2.2 Construct bushfire-prone area feature
# --------------------------
bushfire_sindex = bushfire_gdf.sindex

def bushfire_row_func(row):
    geom = row["geometry"]
    idx = list(bushfire_sindex.intersection(geom.bounds))
    if not idx:
        return {"PFI": row["PFI"], "is_prone": 0}

    cands = bushfire_gdf.iloc[idx]
    is_prone = 1 if cands.geometry.intersects(geom).any() else 0
    return {"PFI": row["PFI"], "is_prone": is_prone}

bushfire_features = produce_geo_features(
    station_property_gdf,
    bushfire_row_func,
    desc="Processing bushfire prone areas"
)
hf_manager.save_gdf(bushfire_features)

In [ ]:
# --------------------------
# 2.3 Construct fire management zone features with one-hot list
# --------------------------
fire_manage_sindex = fire_manage_gdf.sindex

def fire_zone_row_func(row):
    geom = row["geometry"]
    idx = list(fire_manage_sindex.intersection(geom.bounds))

    zonetype = 0
    if idx:
        centroid = geom.centroid
        cands = fire_manage_gdf.iloc[idx]
        hit = cands[cands.geometry.contains(centroid)]
        if not hit.empty:
            zonetype = int(hit["zonetype"].iloc[0])

    one_hot = [0] * 5
    if 0 <= zonetype < 5:
        one_hot[zonetype] = 1

    return {"PFI": row["PFI"], "fire_zonetype": zonetype, "one-hot zone": one_hot}

fire_zone_features = produce_geo_features(
    station_property_gdf,
    fire_zone_row_func,
    desc="Processing fire management zones"
)
hf_manager.save_gdf(fire_zone_features)

In [ ]:
# --------------------------
# 2.4 Construct native vegetation feature
#  One-hot encode vegetation types + retain area to reflect fuel load differences
# --------------------------
native_veg_sindex = native_veg_gdf.sindex

KNOWN_EVC_MUT = ["mosaic", "EVC", "complex", "aggregate", "Wetland-ge", "no veg", "no EVC id."]
KNOWN_EVC_BCS = ["E", "LC", "V", "D", "R", "na"]
KNOWN_XGROUP = [
    "Riparian Scrubs or Swampy Scrubs and Woodlands", "Mallee", "Dry Forests",
    "Lower Slopes or Hills Woodlands", "Riverine Grassy Woodlands or Forests",
    "Montane Grasslands, Shrublands or Woodlands", "Wet or Damp Forests",
    "Plains Woodlands or Forests", "Herb-rich Woodlands",
    "Plains Grasslands and Chenopod Shrublands", "Lowland Forests",
    "Box Ironbark Forests or dry/lower fertility Woodlands", "Wetlands",
    "Heathy Woodlands", "Rocky Outcrop or Escarpment Scrubs",
    "Sub-alpine Grasslands, Shrublands or Woodlands", "Heathlands",
    "Rainforests", "Salt-tolerant and/or succulent Shrublands",
    "Coastal Scrubs Grasslands and Woodlands", "No native vegetation recorded"
]

def _safe_onehot(value, known_list, fallback):
    v = value if value in known_list else fallback
    return [1 if cat == v else 0 for cat in known_list]

def native_veg_row_func(row):
    geom = row["geometry"]
    idx = list(native_veg_sindex.intersection(geom.bounds))

    evc_mut = "no EVC id."
    evc_bcs = "na"
    xgroupname = "No native vegetation recorded"
    areasqm = 0.0

    if idx:
        cands = native_veg_gdf.iloc[idx].copy()
        cands["overlap_area"] = cands.geometry.intersection(geom).area
        cands = cands[cands["overlap_area"] > 0]
        if not cands.empty:
            max_veg = cands.sort_values("overlap_area", ascending=False).iloc[0]
            evc_mut = max_veg.get("evc_mut", "no EVC id.")
            evc_bcs = max_veg.get("evc_bcs", "na")
            xgroupname = max_veg.get("xgroupname", "No native vegetation recorded")
            areasqm = float(max_veg["overlap_area"])

    evc_mut = evc_mut if evc_mut in KNOWN_EVC_MUT else "no veg"
    evc_bcs = evc_bcs if evc_bcs in KNOWN_EVC_BCS else "na"
    xgroupname = xgroupname if xgroupname in KNOWN_XGROUP else "No native vegetation recorded"

    return {
        "PFI": row["PFI"],
        "veg_area": areasqm,
        "evc_mut": _safe_onehot(evc_mut, KNOWN_EVC_MUT, "no veg"),
        "evc_bcs": _safe_onehot(evc_bcs, KNOWN_EVC_BCS, "na"),
        "xgroupname": _safe_onehot(xgroupname, KNOWN_XGROUP, "No native vegetation recorded"),
    }

native_veg_features = produce_geo_features(
    station_property_gdf,
    native_veg_row_func,
    desc="Processing native vegetation"
)

hf_manager.save_gdf(native_veg_features)

In [ ]:
# --------------------------
# 2.5 Construct fire history feature
# Use non-high-risk historical data as features to avoid data leakage
# --------------------------
fire_history_sindex = fire_history_gdf.sindex

def fire_history_row_func(row, buffer_distance=0.1):
    pfi = row["PFI"]
    geom = row["geometry"]
    geom_buffered = geom.buffer(buffer_distance)

    idx = list(fire_history_sindex.intersection(geom_buffered.bounds))

    # No fire records → default 100
    fire_count = 0
    yrs_since_burn = 100

    if idx:
        cands = fire_history_gdf.iloc[idx]
        hit = cands[cands.geometry.intersects(geom)].copy()

        if not hit.empty:
            best = hit.loc[hit["allcount"].idxmax()]
            fire_count = int(best["allcount"])
            yrs_since_burn = float(best["yrsfrburn"])

    return {
        "PFI": pfi,
        "fire_count": fire_count,
        "yrs_since_last_burn": yrs_since_burn,
    }

fire_history_features = produce_geo_features(
    station_property_gdf,
    lambda row: fire_history_row_func(row, buffer_distance=0.1),
    desc="Processing fire history"
)

hf_manager.save_gdf(fire_history_features)

In [ ]:
fire_history_features["yrs_since_last_burn"] = fire_history_features["yrs_since_last_burn"].fillna(100).astype(int)
hf_manager.save_gdf(fire_history_features)

In [ ]:
import re
def parse_onehot_list_fast(onehot_str) -> np.ndarray:
    """
    Fast parser for [0,1,0] or [0 1 0] format
    """
    return np.array(re.findall(r'\d', str(onehot_str)), dtype=int)

def onehot2cols(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    print(f"Processing one-hot for column: {col}")
    tqdm.pandas(desc=f"Parsing {col}")
    mat = np.vstack(df[col].progress_apply(parse_onehot_list_fast).values)
    out_df = pd.DataFrame(
        mat,
        index=df.index,
        columns=[f"{prefix}{i}" for i in range(mat.shape[1])]
    )
    print(f"Completed {col}, generated {out_df.shape[1]} columns\n")
    return out_df

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import RobustScaler
tqdm.pandas()

# --------------------------
# Step 2: Merge all features and standardize to get x_vector
# --------------------------
fire_facility = hf_manager.load('fire_facility_features.parquet')
bushfire      = hf_manager.load('bushfire_features.parquet')
fire_zone     = hf_manager.load('fire_zone_features.parquet')
native_veg    = hf_manager.load('native_veg_features.parquet')
fire_history  = hf_manager.load('fire_history_features.parquet')

# ---parse one-hot ---
zone_df = onehot2cols(fire_zone,'one-hot zone','type')
evc_mut_df = onehot2cols(native_veg, 'evc_mut','evc_mut_')
evc_bcs_df = onehot2cols(native_veg, 'evc_bcs','evc_bcs_')
xgroup_df = onehot2cols(native_veg, 'xgroupname','xgroup_')

veg_df = native_veg[['PFI', 'veg_area']].join([evc_mut_df, evc_bcs_df, xgroup_df])
x_merged = fire_facility.merge(bushfire, on='PFI', how='left')
x_merged = x_merged.merge(zone_df, left_index=True, right_index=True)
x_merged = x_merged.merge(veg_df, on='PFI', how='left')
x_merged = x_merged.merge(fire_history, on='PFI', how='left')
x_vector_unnormalized = x_merged.copy()

x_merged['fire_count_log'] = np.log1p(x_merged['fire_count'])
x_merged['veg_area_log'] = np.log1p(x_merged['veg_area'])
x_merged['closest_facility_distance_log'] = np.log1p(x_merged['closest_facility_distance'])
x_merged['yrs_since_last_burn_log'] = np.log1p(x_merged['yrs_since_last_burn'])

numeric_features = [
    'total_facilities_5km',
    'closest_facility_distance_log',  # log
    'veg_area_log',                   # log
    'fire_count_log',                # log
    'yrs_since_last_burn_log' # log
]

scaler = RobustScaler()
x_merged[numeric_features] = scaler.fit_transform(x_merged[numeric_features])
x_merged = x_merged.drop(columns=[
    'fire_count',
    'veg_area',
    'closest_facility_distance',
    'yrs_since_last_burn'
])

x_vector = x_merged.copy()
hf_manager.save_gdf(x_vector, file_name="x_vector", format="parquet")
hf_manager.save_gdf(x_vector_unnormalized, file_name="x_vector_unnormalized", format="parquet")

In [ ]:
renewable_gdf= hf_manager.load('renewable_gdf.parquet')
station_property_gdf= hf_manager.load('station_property_gdf.parquet')

In [ ]:
# --------------------------
# Step 3: Generate y labels (high-risk + low-risk)
# --------------------------

# Low-risk labels from renewable points (PFI indexed)
renewable_sindex = renewable_gdf.sindex
low_risk_data = []

for _, prop_row in station_property_gdf.iterrows():
    pfi = prop_row["PFI"]
    property_geom = prop_row.geometry

    candidate_indices = list(renewable_sindex.intersection(property_geom.bounds))
    candidate_renewable = renewable_gdf.iloc[candidate_indices] if candidate_indices else gpd.GeoDataFrame()

    has_renewable = 0
    if not candidate_renewable.empty:
        has_renewable = candidate_renewable.geometry.within(property_geom).any()

    low_risk_data.append({
        "PFI": pfi,
        "is_low_risk": int(has_renewable)
    })

low_risk_labels = pd.DataFrame(low_risk_data)

# High-risk labels
x_vector_unnormalized = hf_manager.load('x_vector_unnormalized.parquet')
df = x_vector_unnormalized
mask = (df['fire_count'] >= 4) & (df['is_prone'] == 1)

high_risk_labels = pd.DataFrame({
    "PFI": df["PFI"],
    "is_high_risk": mask.astype(int)
})
# Merge into y_labels (index = PFI)
y_labels = high_risk_labels.merge(low_risk_labels, on="PFI", how="left").fillna(0)
y_labels= y_labels[y_labels["is_high_risk"] != y_labels["is_low_risk"]]
hf_manager.save_gdf(y_labels, file_name="y_labels", format="parquet")